[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AjitKumar-git/cerebro_spheroid_v5/blob/main/colab/CerebroSpheroid_V5_Compact_Growth_SFT.ipynb)

# 🧠 CerebroSpheroid V5: Endogenous Model Growth & Progressive SFT
### Net2Net Subspace Expansion (v5_small 102M ➔ v5_compact 246M) • Freeze & Differential Annealing • Scheduled Deliberation ($P=0.50$) • Google Colab T4/L4/A100

This notebook executes the **Endogenous Spheroid Growth Engine**:
- **🧬 Net2Net Subspace Expansion**: Expands `v5_small` (102.4M params: 16 shells, $d=640$, $r=256$, 40 tracts) into `v5_compact` (245.6M params: 32 shells, $d=768$, $r=512$, 96 tracts) preserving **100% of the 900k-step TPU linguistic and biological priors**.
- **🔒 Phase 1 (Subspace Isolation)**: Core inherited weights are frozen for initial steps while new outer channels, 56 new shortcut tracts, and radial routing adapt to the data.
- **🔓 Phase 2 (Differential Cosine Annealing)**: Full model unfreezes with differential learning rates ($0.1\times$ on core, $1.0\times$ on expansion) to prevent catastrophic forgetting.
- **💭 Scheduled Deliberation ($P=0.50$)**: 50% internal thought feedback via motor efference / 50% teacher-forced ground truth for emergent System 2 scratchpad reasoning.

--- 
## 🚀 Step 1: Hardware Verification & Accelerator Setup

In [ ]:
import os, sys, torch

print("=" * 80)
print("⚡ CEREBROSPHEROID V5: ACCELERATOR ENVIRONMENT CHECK")
print("=" * 80)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    bf16_ok = torch.cuda.is_bf16_supported()
    print(f"✓ GPU Detected       : {gpu_name}")
    print(f"✓ Dedicated VRAM     : {vram_gb:.2f} GB")
    print(f"✓ Native BFloat16    : {'Supported (Recommended)' if bf16_ok else 'Fallback to FP16'}")
    
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
else:
    print("⚠️ No GPU detected! Please navigate to: Runtime -> Change runtime type -> T4 GPU / L4 GPU.")

--- 
## 📦 Step 2: Clone Repository & Install Dependencies

In [ ]:
import os, subprocess

# Clone repository if running in fresh Colab session
if not os.path.exists("src/spheroid_v5.py"):
    !git clone https://github.com/AjitKumar-git/cerebro_spheroid_v5.git
    %cd cerebro_spheroid_v5

!pip install -q torch transformers datasets sentencepiece pandas accelerate

--- 
## 📥 Step 3: Download / Verify Base Foundation Checkpoint (`v5_small`)

In [ ]:
import os, urllib.request

os.makedirs("checkpoints", exist_ok=True)
os.makedirs("data/tokenizer", exist_ok=True)

small_ckpt_candidates = [
    "checkpoints/v5_small_latest_step920k.pt",
    "checkpoints/v5_small_best.pt",
    "checkpoints/v5_small_latest_step900k.pt"
]

base_ckpt = next((p for p in small_ckpt_candidates if os.path.exists(p)), None)

if not base_ckpt:
    print("🔄 Downloading pre-trained Step 900k foundation weights...")
    # Direct mirror from Bioflo Cloud Storage or HuggingFace
    url = "https://bioflo.in/api/cerebro/download.php?model=v5_small_latest_step920k.pt"
    target_file = "checkpoints/v5_small_latest_step920k.pt"
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "CerebroColab/5.0"})
        with urllib.request.urlopen(req) as resp, open(target_file, 'wb') as out_f:
            out_f.write(resp.read())
        base_ckpt = target_file
        print(f"✓ Downloaded: {base_ckpt}")
    except Exception as e:
        print(f"Note: If direct download fails, please upload your 'v5_small_latest_step920k.pt' to checkpoints/")
        base_ckpt = "checkpoints/v5_small_latest_step920k.pt"
else:
    print(f"✓ Found existing foundation checkpoint: {base_ckpt}")

--- 
## 🧬 Step 4: Endogenous Model Expansion (102M ➔ 246M)
This step expands the parameter matrices using **Function-Preserving Net2Net subspace mapping**:
- Surface channels: $640 \to 768$ (zero-padded output projections)
- Radial wave routing: $256 \to 512$ channels
- Concentric laminae: $16 \to 32$ shells (interpolated)
- Axonal tracts: $40 \to 96$ shortcut pathways

In [ ]:
from scripts.train_v5_compact_growth_sft import expand_v5_small_to_compact

expanded_seed_ckpt = "checkpoints/v5_compact_expanded_warmstart.pt"
compact_cfg_path = "configs/v5_compact_250m.json"

model, cfg = expand_v5_small_to_compact(
    small_ckpt_path=base_ckpt,
    compact_cfg_path=compact_cfg_path,
    output_path=expanded_seed_ckpt,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

total_p = sum(p.numel() for p in model.parameters())
print(f"✨ v5_compact successfully initialized with {total_p:,} parameters ({total_p/1e6:.1f}M)!")

--- 
## 🔥 Step 5: Launch Freeze & Annealing SFT ($P=0.50$ Scheduled Deliberation)

We execute **Two-Phase Progressive SFT**:
- **Phase 1 (First 300 steps)**: Core 102M parameters remain **FROZEN** while new channels and tracts adapt.
- **Phase 2 (Steps 300 - 1500)**: Full model unfreezes with **Differential Cosine Annealing** ($0.1\times$ core LR, $1.0\times$ expansion LR).
- **$P=0.50$ Scheduled Deliberation**: Trains both deep scratchpad `<think>` reasoning and high-speed direct answering.

In [ ]:
# Execute the growth training pipeline directly
!python scripts/train_v5_compact_growth_sft.py \
    --small-ckpt "checkpoints/v5_small_latest_step920k.pt" \
    --compact-config "configs/v5_compact_250m.json" \
    --output-ckpt "checkpoints/v5_compact_growth_sft_p05.pt" \
    --max-steps 1500 \
    --freeze-steps 300 \
    --batch-size 2 \
    --accum-steps 8 \
    --lr 2e-4 \
    --min-lr 2e-6 \
    --sampling-p 0.50 \
    --probe-interval 100

--- 
## 🔍 Step 6: Interactive Generation & Reasoning Probe
Test the converged `v5_compact` model on live math, logic, and factual prompts to inspect the emergence of natural step-by-step reasoning.

In [ ]:
import torch
from src.spheroid_v5 import CerebroSpheroidV5, ModelConfigV5, infer_v5_config_from_state_dict
from src.corpus import build_tokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt_path = "checkpoints/v5_compact_growth_sft_p05.pt"
if not os.path.exists(ckpt_path):
    ckpt_path = "checkpoints/v5_compact_growth_sft_p05_expanded_seed.pt"

print(f"Loading checkpoint: {ckpt_path}")
raw_ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
cfg = raw_ckpt.get("config", ModelConfigV5(**json.load(open("configs/v5_compact_250m.json"))))
model = CerebroSpheroidV5(cfg).to(device)
model.load_state_dict(raw_ckpt["model_state"], strict=False)
model.eval()

tokenizer = build_tokenizer("data/tokenizer")

prompts = [
    "What is 24 * 7?",
    "If a train travels 60 miles per hour, how far does it go in 3.5 hours?",
    "What is the chemical symbol and atomic number of gold?",
    "Solve for x: 3x - 9 = 21."
]

print("=" * 80)
print("🧠 CEREBROSPHEROID V5 COMPACT: LIVE REASONING PROBES")
print("=" * 80)

for p in prompts:
    output = model.generate_text(
        p,
        tokenizer=tokenizer,
        max_new_tokens=96,
        temperature=0.2,
        device=device,
    )
    print(f"\n👉 Prompt: {p}")
    print(f"💡 Response:\n{output.strip()}")
    print("-" * 60)

--- 
## 💾 Step 7: Checkpoint Backup / Google Drive Export

In [ ]:
from google.colab import drive, files
import shutil

# Option A: Direct browser download
best_ckpt = "checkpoints/v5_compact_growth_sft_p05.pt"
if os.path.exists(best_ckpt):
    print(f"Preparing {best_ckpt} for download...")
    # files.download(best_ckpt)

# Option B: Mount Google Drive
# drive.mount('/content/drive')
# shutil.copy(best_ckpt, '/content/drive/MyDrive/v5_compact_growth_sft_p05.pt')
# print("✓ Checkpoint backed up to Google Drive!")